##### ARTI 560 - Computer Vision  
## Image Classification using Transfer Learning - Exercise 

### Objective

In this exercise, you will:

1. Select another pretrained model (e.g., VGG16, MobileNetV2, or EfficientNet) and fine-tune it for CIFAR-10 classification.  
You'll find the pretrained models in [Tensorflow Keras Applications Module](https://www.tensorflow.org/api_docs/python/tf/keras/applications).

2. Before training, inspect the architecture using model.summary() and observe:
- Network depth
- Number of parameters
- Trainable vs Frozen layers

3. Then compare its performance with ResNet and the custom CNN.

### Questions:

- Which model achieved the highest accuracy?
- Which model trained faster?
- How might the architecture explain the differences?

1- Which model achieved the highest accuracy?

EfficientNetV2-B2 (fine-tuned) achieved the highest test accuracy (91.92%)


2- Which model trained faster?

EfficientNetV2-B2 trained faster per epoch than ResNet50V2 due to fewer parameters and fewer trainable layers

3-How might the architecture explain the differences?
| Architecture Feature                  | Effect                                                                                        |
| ------------------------------------- | --------------------------------------------------------------------------------------------- |
| Depth & residual connections (ResNet) | Allows very deep networks to learn, improves fine-tuning accuracy                             |
| Compound scaling (EfficientNetV2-B2)  | Efficiently balances depth, width, and resolution, giving high accuracy with fewer parameters |
| Frozen layers                         | Rapid convergence with pretrained feature extraction                                          |
| Fine-tuned layers                     | Task-specific adaptation improves accuracy slightly                                           |
| Custom CNN (shallow)                  | Limited representation → lower accuracy                                                       |


Summary:

| Model | Params | Accuracy | Training Speed |
|-------|--------|----------|----------------|
| Custom CNN | Smallest | Lowest | Fast |
| ResNet50V2 | Largest (23.5M) | High | Slower |
| EfficientNetV2-B2 | Moderate (8.7M) | Highest | Fastest (per param) |

Deeper architectures and pretrained features are key to high accuracy.

EfficientNetV2-B2 achieves a high accuracy–efficiency tradeoff.

Fine-tuning a few layers is sufficient to maximize performance on CIFAR-10.

In [14]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetV2B2
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input

(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

class_names = [
    "airplane","automobile","bird","cat","deer",
    "dog","frog","horse","ship","truck"
]

y_train = y_train.squeeze().astype("int64")
y_test  = y_test.squeeze().astype("int64")

x_train = x_train.astype("float32")
x_test  = x_test.astype("float32")


In [15]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
], name="augmentation")


In [16]:
effnet_base = EfficientNetV2B2(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)
effnet_base.trainable = False



35839040/35839040 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [17]:
effnet_model = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    data_augmentation,
    layers.Resizing(224, 224, interpolation="bilinear"),
    layers.Lambda(preprocess_input),      # IMPORTANT for EfficientNetV2
    effnet_base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(10)                      # logits
], name="cifar10_efficientnetv2")


In [18]:
effnet_model.summary()


Model: "cifar10_efficientnetv2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ augmentation (Sequential)       │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resizing_1 (Resizing)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda_1 (Lambda)               │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetv2-b2 (Functional)  │ (None, 7, 7, 1408)     │     8,769,374 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1408)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │        14,090 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,783,464 (33.51 MB)

 Trainable params: 14,090 (55.04 KB)

 Non-trainable params: 8,769,374 (33.45 MB)

In [19]:
effnet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

history_eff = effnet_model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=3,
    batch_size=64,
    verbose=1
)


Epoch 1/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 139s 176ms/step - accuracy: 0.7061 - loss: 0.9234 - val_accuracy: 0.9020 - val_loss: 0.2985
Epoch 2/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 121s 171ms/step - accuracy: 0.8169 - loss: 0.5333 - val_accuracy: 0.9078 - val_loss: 0.2752
Epoch 3/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 121s 171ms/step - accuracy: 0.8339 - loss: 0.4830 - val_accuracy: 0.9094 - val_loss: 0.2616


In [20]:
test_loss_eff, test_acc_eff = effnet_model.evaluate(x_test, y_test, verbose=0)
print("EfficientNetV2 (frozen) test accuracy:", test_acc_eff)


EfficientNetV2 (frozen) test accuracy: 0.9165999889373779


In [21]:
effnet_base.trainable = True
for layer in effnet_base.layers[:-15]:
    layer.trainable = False


print("Trainable layers in backbone:",
      sum(l.trainable for l in effnet_base.layers),
      "/", len(effnet_base.layers))

effnet_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

history_eff_ft = effnet_model.fit(
    x_train, y_train,
    validation_split=0.1,
    epochs=3,
    batch_size=64,
    verbose=1
)


Trainable layers in backbone: 15 / 349
Epoch 1/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 151s 189ms/step - accuracy: 0.8201 - loss: 0.5422 - val_accuracy: 0.9052 - val_loss: 0.2881
Epoch 2/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 129s 183ms/step - accuracy: 0.8325 - loss: 0.4967 - val_accuracy: 0.9100 - val_loss: 0.2735
Epoch 3/3
704/704 ━━━━━━━━━━━━━━━━━━━━ 129s 183ms/step - accuracy: 0.8386 - loss: 0.4819 - val_accuracy: 0.9122 - val_loss: 0.2634


In [22]:
test_loss_eff_ft, test_acc_eff_ft = effnet_model.evaluate(x_test, y_test, verbose=0)
print("EfficientNetV2 (fine-tuned) test accuracy:", test_acc_eff_ft)


EfficientNetV2 (fine-tuned) test accuracy: 0.9192000031471252
